In [5]:
!pip install requests
!pip install pandas
!pip install pyproj


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 20.0 MB/s  0:00:00m0:00:010:01

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [6]:
import json
import requests
from pyproj import CRS, Transformer

# Arquivo gerado pelo script da primeira API
with open("../dados/originais/dadosapi_casasbombaebap.json", "r", encoding="utf-8") as arquivo:
    dados = json.load(arquivo)

print("Quantidade de casas de bomba:", len(dados["features"]))

# Procura a referencia espacial no JSON coletado
referencia = dados.get("spatialReference")

if not referencia:
    for casa in dados["features"]:
        referencia = casa.get("geometry", {}).get("spatialReference")
        if referencia:
            break

# Se o JSON nao trouxer a referencia, consulta somente os metadados da camada
# (nao faz nova coleta das casas e nao consulta a API de inundacao).
if not referencia:
    url = "https://gis-smamus.portoalegre.rs.gov.br/server/rest/services/Hosted/Casas_de_Bomba___EBAP_210126/FeatureServer/0"
    resposta = requests.get(url, params={"f": "json"}, timeout=30)
    resposta.raise_for_status()
    metadados = resposta.json()
    if "error" in metadados:
        raise ValueError(metadados["error"])
    referencia = metadados.get("extent", {}).get("spatialReference")

if not referencia:
    raise ValueError("Nao foi possivel identificar o sistema de coordenadas original.")

# Converte a descricao do sistema de coordenadas em um objeto CRS
# Define o sistema de coordenadas original das casas de bomba (TM-POA)
# Utilizamos os parametros de projecao informados pela API,
# pois o WKT retornado apresenta um erro de formatacao.

origem = CRS.from_proj4(
    "+proj=tmerc "
    "+lat_0=0 "
    "+lon_0=-51 "
    "+k=0.999995 "
    "+x_0=300000 "
    "+y_0=5000000 "
    "+ellps=GRS80 "
    "+units=m "
    "+no_defs "
    "+type=crs"
)

# Sistema de destino: longitude e latitude
destino = CRS.from_epsg(4326)

# Cria o conversor de coordenadas
conversor = Transformer.from_crs(origem, destino, always_xy=True)

# EPSG:4326 = longitude e latitude, em graus
conversor = Transformer.from_crs(origem, "EPSG:4326", always_xy=True)
print("Sistema original:", origem.name)

Quantidade de casas de bomba: 24
Sistema original: unknown


In [7]:
# Mantem os atributos e os demais dados de cada casa de bomba
for casa in dados["features"]:
    geometria = casa["geometry"]
    x = geometria["x"]
    y = geometria["y"]

    # Transforma as coordenadas originais em longitude e latitude
    longitude, latitude = conversor.transform(x, y)

    # Confere se o resultado esta na regiao de Porto Alegre
    if not (-52 <= longitude <= -50 and -31 <= latitude <= -29):
        nome = casa.get("attributes", {}).get("name", "sem nome")
        raise ValueError(f"Coordenadas inesperadas para {nome}: {longitude}, {latitude}")

    # Mantem os nomes x e y para que o formato continue igual ao da API 1
    geometria["x"] = round(longitude, 8)
    geometria["y"] = round(latitude, 8)

    # Se existir referencia espacial dentro da geometria, atualiza tambem
    if "spatialReference" in geometria:
        geometria["spatialReference"] = {"wkid": 4326, "latestWkid": 4326}

# Atualiza o sistema de coordenadas indicado no cabecalho do JSON
# e remove metadados antigos que possam descrever coordenadas projetadas.
dados["spatialReference"] = {"wkid": 4326, "latestWkid": 4326}

# Cria um NOVO arquivo. O JSON original fica intacto.
with open("../dados/processados/api_casasbombaebap_convertido.json", "w", encoding="utf-8") as arquivo:
    json.dump(dados, arquivo, ensure_ascii=False, indent=4)

print("JSON convertido salvo com sucesso!")
print("Quantidade de casas de bomba:", len(dados["features"]))
print("Exemplo de coordenadas:", dados["features"][0]["geometry"])

JSON convertido salvo com sucesso!
Quantidade de casas de bomba: 24
Exemplo de coordenadas: {'x': -51.2143618, 'y': -30.0160068}
